**Viterbi algorithm for Nature Primer**

You have learned how the Viterbi algorithm can be used in Hidden Markov Models (HMMs) to find the most probable sequence of hidden states that result in an observed sequence (such as DNA). The task is to implement the Viterbi algorithm using a simplified gene model as described in the Nature Primer.

1. Define Model Parameters

 Define the following components of your HMM:

 States: e.g., E (Exon), I (Intron)

 Transition Probabilities: e.g., P(I|E), P(E|I), etc.

 Emission Probabilities: e.g., P(C|E), P(G|I), etc.

2. Log Probability Calculation Function

 Write a function that computes the log-probability of an observed sequence given a path of states:

3. Implement the Viterbi Algorithm

 Implement the Viterbi algorithm to find the most probable state sequence for a given DNA sequence using the defined HMM.

Input:

A state sequence string (e.g., "EEEEEEEEEEEEEEEEEE5IIIIIII")

A DNA sequence string (e.g., "CTTCATGTGAAAGCAGACGTAAGTCA")

In [5]:
import math

states = ['E', '5', 'I']
start_prob = {'E': 1.0, '5': 0.0, 'I': 0.0}

trans_prob = {'E': {'E': 0.9, '5': 0.1}, '5': {'I': 1.0}, 'I': {'I': 0.9}}

emit_prob = {'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25}, '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0}, 'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}}

def log(x):
    return -math.inf if x == 0 else math.log(x)

def log_prob_of_a_path(path: str, seq: str) -> float:
    if len(path) != len(seq):
        print("Lengths of path and sequence are not equal.")
        return -1

    prob = 0.0
    for i in range(len(seq)):
        s = path[i]
        o = seq[i]
        if i == 0:
            prob += log(start_prob[s])
        else:
            prob += log(trans_prob[path[i-1]][s])
        prob += log(emit_prob[s][o])

    last_state = path[-1]
    if last_state == 'I':
        prob += log(0.1)

    return round(prob, 4)

def viterbi(sequence):

    dp = [{} for _ in range(len(sequence))]
    backpointer = [{} for _ in range(len(sequence))]

    for s in states:
        dp[0][s] = log(start_prob[s]) + log(emit_prob[s][sequence[0]])
        backpointer[0][s] = None

    for t in range(1, len(sequence)):
        for curr_state in states:
            max_prob = -math.inf
            max_state = None

            for prev_state in states:
                if curr_state in trans_prob.get(prev_state, {}):
                    prob = dp[t-1][prev_state] + log(trans_prob[prev_state][curr_state])
                    if prob > max_prob:
                        max_prob = prob
                        max_state = prev_state

            # If a valid transition exists
            if max_state is not None:
                dp[t][curr_state] = max_prob + log(emit_prob[curr_state][sequence[t]])
                backpointer[t][curr_state] = max_state
            else:
                dp[t][curr_state] = -math.inf

    max_prob = -math.inf
    max_state = None
    for s in states:
        end_trans_prob = 0.1 if s == 'I' else 0
        prob = dp[len(sequence)-1][s]
        if end_trans_prob > 0:
            prob += log(end_trans_prob)

        if prob > max_prob:
            max_prob = prob
            max_state = s

    best_path = [max_state]
    for t in range(len(sequence)-1, 0, -1):
        max_state = backpointer[t][max_state]
        best_path.insert(0, max_state)

    return round(max_prob, 4), ''.join(best_path)


sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
path = "EEEEEEEEEEEEEEEEEE5IIIIIII"

print("Log probability of given path: ", log_prob_of_a_path(path, sequence))

best_log_prob, most_likely_path = viterbi(sequence)
print("Viterbi best log probability: ", best_log_prob)
print("Most likely path: ", most_likely_path)

Log probability of given path:  -41.2197
Viterbi best log probability:  -38.6777
Most likely path:  EEEEEEEEEEEEEEEEEEEEEEEEEE
